# 02：数据清洗、变量构造和合并

本 notebook 完成：
1. 读取并拼接不同时间段的资产负债表和利润表
2. 主键统一：股票代码 6 位字符串 `code`，会计年度整数 `year`
3. 合并 4 个数据源：资产负债表 + 利润表 + 股权结构 + 公司信息
4. 构造 13 个分析变量
5. 缺失值检查、异常值处理、缩尾
6. 输出 CSV + Parquet + 变量字典 + 统计表

In [ ]:
import os, glob
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_ROOT = os.getcwd()
raw_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
dict_dir = os.path.join(PROJECT_ROOT, 'data', 'dict')
clean_dir = os.path.join(PROJECT_ROOT, 'data', 'clean')
combined_dir = os.path.join(PROJECT_ROOT, 'data', 'combined')
output_dir = os.path.join(PROJECT_ROOT, 'output', 'tables')

for d in [dict_dir, clean_dir, combined_dir, output_dir]:
    os.makedirs(d, exist_ok=True)

log_lines = []
def log(msg):
    print(msg)
    log_lines.append(f'{datetime.now().isoformat()}: {msg}')

log('=== 02_clean_construct_variables START ===')

## 1. 读取原始数据文件

CSMAR 数据分布在 4 个来源、两个时间段（2000-2010 / 2011-2024），需要先按来源拼接再合并。

### 数据文件路径映射

| 来源 | 2000-2010 | 2011-2024 |
|------|-----------|-----------|
| 资产负债表 | `资产负债表-2000-2010/跨表查询...xlsx` | `资产负债表-2011-2024/跨表查询...xlsx` |
| 利润表+现金流量表 | `利润表-现金流量表-2000-2010/跨表查询...xlsx` | `利润表-现金流量表-2011-2024/跨表查询...xlsx` |
| 常用变量 | `CSMAR常用变量-2000-2024/常用变量查询（年度）.xlsx`（单文件） | — |
| 公司信息 | `上市公司基本信息年度表/STK_LISTEDCOINFOANL.xlsx`（单文件） | — |

In [ ]:
def find_file(pattern):
    """Find a file by pattern in data/raw/ subdirectories."""
    matches = glob.glob(os.path.join(raw_dir, '**', pattern), recursive=True)
    if not matches:
        raise FileNotFoundError(f'Pattern not found: {pattern}')
    return matches[0]

file_bs_0010 = find_file('*资产负债表-2000-2010*/跨表查询*.xlsx')
file_bs_1124 = find_file('*资产负债表-2011-2024*/跨表查询*.xlsx')
file_is_0010 = find_file('*利润表-现金流量表-2000-2010*/跨表查询*.xlsx')
file_is_1124 = find_file('*利润表-现金流量表-2011-2024*/跨表查询*.xlsx')
file_vars = find_file('*常用变量*/常用变量查询*.xlsx')
file_info = find_file('*基本信息年度表*/STK_LISTEDCOINFOANL.xlsx')

print('所有文件定位成功。')
for f in [file_bs_0010, file_bs_1124, file_is_0010, file_is_1124, file_vars, file_info]:
    rel = os.path.relpath(f, PROJECT_ROOT)
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {rel} ({size_mb:.1f} MB)')

## 2. 读取并拼接资产负债表（2000-2010 + 2011-2024）

CSMAR 的 xlsx 文件中，第 1 行是中文标签，第 2 行是单位。需要跳过前两行，用代码列名。

In [ ]:
def read_csmar_xlsx(filepath):
    """Read CSMAR xlsx file.
    
    CSMAR xlsx files have a consistent structure:
    - Row 0: CSMAR column codes (e.g., FS_Combas-A001000000)
    - Row 1: Chinese labels (e.g., 资产总计)
    - Row 2: Units (e.g., 元, 没有单位)
    - Row 3+: Data
    
    Returns DataFrame with CSMAR codes as column names.
    """
    df_raw = pd.read_excel(filepath, header=None)
    df_raw.columns = df_raw.iloc[0]  # Use row 0 (CSMAR codes) as column names
    df = df_raw.iloc[3:].copy()       # Data starts from row 3
    df = df.reset_index(drop=True)
    return df

# Read and concatenate
bs1 = read_csmar_xlsx(file_bs_0010)
bs2 = read_csmar_xlsx(file_bs_1124)
bs = pd.concat([bs1, bs2], ignore_index=True)
log(f'Balance sheet: {len(bs1)} + {len(bs2)} = {len(bs)} rows')
print(f'Columns: {bs.columns.tolist()}')

In [ ]:
# Inspect first few rows
print(f'Shape: {bs.shape}')
print(f'\nFirst 5 rows of key columns:')
display(bs[['code', 'EndDate', 'FS_Combas-A001000000', 'FS_Combas-A002000000']].head())
print(f'\nData types sample:')
print(bs[['code', 'EndDate']].dtypes)

## 3. 读取并拼接利润表（2000-2010 + 2011-2024）

In [ ]:
is1 = read_csmar_xlsx(file_is_0010)
is2 = read_csmar_xlsx(file_is_1124)
income_stmt = pd.concat([is1, is2], ignore_index=True)
log(f'Income statement: {len(is1)} + {len(is2)} = {len(income_stmt)} rows')
print(f'Columns: {income_stmt.columns.tolist()}')

## 4. 读取股权结构（常用变量）

In [ ]:
vars_df = read_csmar_xlsx(file_vars)
log(f'Common variables: {len(vars_df)} rows')
print(f'Ownership columns: {[c for c in vars_df.columns if "Shr" in c or "shrcr" in c.lower() or "shrhfd" in c.lower()]}')

## 5. 读取公司基本信息（行业分类 + 上市日期）

In [ ]:
info_df = read_csmar_xlsx(file_info)
log(f'Company info: {len(info_df)} rows')
print(f'Key columns: {[c for c in info_df.columns if "Symbol" in c or "Industry" in c or "EndDate" in c or "list" in c.lower()][:15]}')

## 6. 标准化主键和列名

In [ ]:
def standardize_panel(df, code_col='code', year_col='EndDate'):
    """Standardize code to 6-digit string, year to integer, filter >=2000."""
    df = df.copy()
    df['code'] = df[code_col].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6).str.strip()
    # Handle both 'YYYY' and 'YYYY-MM-DD' date formats
    y = pd.to_datetime(df[year_col], errors='coerce')
    if y.notna().sum() > 0:
        df['year'] = y.dt.year
    else:
        df['year'] = pd.to_numeric(df[year_col], errors='coerce')
    before = len(df)
    df = df[df['year'] >= 2000].copy()
    df['year'] = df['year'].astype(int)
    print(f'  Filter year>=2000: {before} -> {len(df)} rows')
    return df

bs = standardize_panel(bs)
income_stmt = standardize_panel(income_stmt)
log(f'BS after standardize: {len(bs)} rows, {bs["code"].nunique()} firms')

## 7. 合并资产负债表 + 利润表 → 构造财务变量

In [ ]:
# Keep only needed columns from balance sheet
bs_cols_map = {
    'FS_Combas-A001000000': 'total_assets',       # 资产总计
    'FS_Combas-A002000000': 'total_liab',         # 负债合计
    'FS_Combas-A002100000': 'current_liab',       # 流动负债合计
    'FS_Combas-A002200000': 'noncurrent_liab',    # 非流动负债合计
    'FS_Combas-A001101000': 'cash_equiv',         # 货币资金
    'FS_Combas-A002101000': 'short_loan',         # 短期借款
    'FS_Combas-A002201000': 'long_loan',          # 长期借款
    'FS_Combas-A003000000': 'equity',             # 所有者权益合计
}

bs_sub = bs[['code', 'year'] + list(bs_cols_map.keys())].copy()
bs_sub = bs_sub.rename(columns=bs_cols_map)

# Keep only needed columns from income statement
is_cols_map = {
    'FS_Comins-B002000000': 'net_profit',  # 净利润
}
is_sub = income_stmt[['code', 'year'] + list(is_cols_map.keys())].copy()
is_sub = is_sub.rename(columns=is_cols_map)

# Merge
panel = bs_sub.merge(is_sub, on=['code', 'year'], how='left')
log(f'BS + IS merged: {len(panel)} rows')

# Convert to numeric
for c in list(bs_cols_map.values()) + list(is_cols_map.values()):
    panel[c] = pd.to_numeric(panel[c], errors='coerce')

print(f'Panel shape: {panel.shape}')

## 8. 构造财务比率变量

In [ ]:
# Lev = total_liab / total_assets
panel['Lev_raw'] = panel['total_liab'] / panel['total_assets']

# SL = current_liab / total_assets
panel['SL_raw'] = panel['current_liab'] / panel['total_assets']

# LL = noncurrent_liab / total_assets
panel['LL_raw'] = panel['noncurrent_liab'] / panel['total_assets']

# SDR = current_liab / total_liab
panel['SDR_raw'] = panel['current_liab'] / panel['total_liab']

# Cash = cash_equiv / total_assets
panel['Cash_raw'] = panel['cash_equiv'] / panel['total_assets']

# ROA = net_profit / total_assets
panel['ROA_raw'] = panel['net_profit'] / panel['total_assets']

# ROE = net_profit / equity
panel['ROE_raw'] = panel['net_profit'] / panel['equity']

# SLoan = short_loan / total_assets
panel['SLoan_raw'] = panel['short_loan'] / panel['total_assets']

# LLoan = long_loan / total_assets
panel['LLoan_raw'] = panel['long_loan'] / panel['total_assets']

# Size = ln(total_assets)
panel['Size'] = np.log(panel['total_assets'].clip(lower=1))

log(f'Constructed {10} financial ratios')
print('Financial ratios constructed.')

## 9. 合并股权结构数据（Top1, HHI5）

In [ ]:
# Standardize ownership data
own = standardize_panel(vars_df, code_col='Stkcd', year_col='accper')

own_cols_map = {
    'Shrcr1': 'Top1',       # 第一大股东持股比例
    'Shrhfd5': 'HHI5_raw',  # 前五大股东持股集中度
}

own_sub = own[['code', 'year'] + list(own_cols_map.keys())].copy()
own_sub = own_sub.rename(columns=own_cols_map)

# Convert to numeric
for c in own_cols_map.values():
    own_sub[c] = pd.to_numeric(own_sub[c], errors='coerce')

# Top1 in CSMAR is already in percentage (e.g., 35.5 means 35.5%), convert to decimal
if own_sub['Top1'].max() > 1:
    own_sub['Top1'] = own_sub['Top1'] / 100
    print('Top1 converted from % to decimal.')

# HHI5 in CSMAR: sum of squared shares. If values > 1, they're in %^2 units, convert
if own_sub['HHI5_raw'].max() > 1:
    # CSMAR HHI5 is sum of squares of percentage shares (e.g., 30%→900, 20%→400, sum=1300+...)
    # Convert to decimal: HHI5_decimal = HHI5_raw / 10000
    own_sub['HHI5_raw'] = own_sub['HHI5_raw'] / 10000
    print('HHI5 converted to decimal.')

before = len(panel)
panel = panel.merge(own_sub, on=['code', 'year'], how='left')
log(f'Ownership merged: {before} -> {len(panel)} rows')
print(f'Top1 available: {panel["Top1"].notna().sum()}/{len(panel)}')

## 10. 合并公司基本信息（行业 + 上市日期）

In [ ]:
# Company info: one row per firm per year (EndDate), need to extract listing date and industry
info = read_csmar_xlsx(file_info)

# Standardize code
info['code'] = info['Symbol'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6).str.strip()

# Extract industry code (first letter for industry category)
if 'IndustryCode' in info.columns:
    info['industry_code'] = info['IndustryCode'].astype(str).str.strip()
    info['industry_cat'] = info['industry_code'].str[0].str.upper()
    print(f'Industry categories: {info["industry_cat"].value_counts().to_dict()}')

# Company name
if 'ShortName' in info.columns:
    info['company_name'] = info['ShortName'].astype(str)

# EndDate in company info is 'YYYY-MM-DD' format, parse as datetime then extract year
info['_date'] = pd.to_datetime(info['EndDate'], errors='coerce')
info['info_year'] = info['_date'].dt.year
info_list_year = info.groupby('code')['info_year'].min().reset_index()
info_list_year.columns = ['code', 'list_year']

# Get company name and industry from the most recent record
info_latest = info.sort_values('info_year').groupby('code').last().reset_index()
info_merge = info_latest[['code', 'industry_code', 'industry_cat', 'company_name']].copy()
info_merge = info_merge.merge(info_list_year, on='code', how='left')

# Merge into panel
before = len(panel)
panel = panel.merge(info_merge, on='code', how='left')
log(f'Company info merged: {before} -> {len(panel)} rows')

# Age = year - list_year + 1
if 'list_year' in panel.columns:
    panel['Age'] = panel['year'] - panel['list_year'] + 1
    panel.loc[panel['Age'] < 0, 'Age'] = None
    print(f'Age constructed. Range: {panel["Age"].min():.0f} - {panel["Age"].max():.0f}')

## 11. 重复值检查与处理

In [ ]:
dups = panel[panel.duplicated(subset=['code', 'year'], keep=False)]
n_dup = len(dups)
log(f'Duplicate code-year pairs: {n_dup}')

if n_dup > 0:
    # Keep observation with fewer missing values
    ratio_cols = [c for c in panel.columns if c.endswith('_raw')]
    panel['_nmiss'] = panel[ratio_cols].isna().sum(axis=1) if ratio_cols else 0
    panel = panel.sort_values('_nmiss').drop_duplicates(subset=['code', 'year'], keep='first')
    panel = panel.drop(columns=['_nmiss'])
    log(f'After dedup: {len(panel)} rows')

print(f'Final unique code-year: {len(panel)}')

## 12. 缺失值统计

In [ ]:
ratio_cols = [c for c in panel.columns if c.endswith('_raw')] + ['Top1']
ratio_cols = [c for c in ratio_cols if c in panel.columns]

missing_stats = []
for c in ratio_cols:
    n_miss = panel[c].isna().sum()
    pct = n_miss / len(panel) * 100
    missing_stats.append({'variable': c, 'missing_count': n_miss, 'missing_pct': round(pct, 2)})
    if pct > 0:
        print(f'  {c}: {n_miss} missing ({pct:.1f}%)')

pd.DataFrame(missing_stats).to_csv(os.path.join(output_dir, 'missing_summary.csv'), index=False)
log('Missing summary saved')

## 13. 异常值检查与处理

In [ ]:
# Check for invalid ratios
checks = [
    ('Lev > 1', 'Lev_raw', lambda x: x > 1),
    ('Lev < 0', 'Lev_raw', lambda x: x < 0),
    ('SDR > 1', 'SDR_raw', lambda x: x > 1),
    ('SDR < 0', 'SDR_raw', lambda x: x < 0),
    ('Cash > 1', 'Cash_raw', lambda x: x > 1),
    ('Cash < 0', 'Cash_raw', lambda x: x < 0),
    ('total_assets <= 0', 'total_assets', lambda x: x <= 0),
    ('equity <= 0', 'equity', lambda x: x <= 0),
]

for label, col, cond in checks:
    if col in panel.columns:
        mask = cond(panel[col])
        n = mask.sum()
        if n > 0:
            print(f'  {label}: {n} observations -> set to NaN')
            # Invalidate affected ratios
            affected = [c for c in ratio_cols if c in panel.columns]
            for c in affected:
                panel.loc[mask, c] = np.nan

log('Outlier check complete')

## 14. 缩尾处理（Winsorize）

对以下比率变量按年度在 1%/99% 分位数缩尾。Size、Top1、Age 不做缩尾。

In [ ]:
winsor_vars = ['Lev_raw', 'SL_raw', 'LL_raw', 'SDR_raw', 'Cash_raw',
               'ROA_raw', 'ROE_raw', 'SLoan_raw', 'LLoan_raw', 'HHI5_raw']
winsor_vars = [v for v in winsor_vars if v in panel.columns]

winsor_stats = []

for v in winsor_vars:
    clean_name = v.replace('_raw', '')
    series = panel[v].dropna()
    
    # Pre-winsor stats
    om, os_std = series.mean(), series.std()
    omin, omax = series.min(), series.max()
    
    # Winsorize by year
    panel[clean_name] = panel.groupby('year')[v].transform(
        lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99))
    )
    
    # Post-winsor stats
    w_series = panel[clean_name].dropna()
    wm, ws_std = w_series.mean(), w_series.std()
    wmin, wmax = w_series.min(), w_series.max()
    
    winsor_stats.append({
        'variable': clean_name,
        'orig_mean': round(om, 6), 'orig_std': round(os_std, 6),
        'orig_min': round(omin, 6), 'orig_max': round(omax, 6),
        'win_mean': round(wm, 6), 'win_std': round(ws_std, 6),
        'win_min': round(wmin, 6), 'win_max': round(wmax, 6),
    })
    log(f'Winsorized {v} -> {clean_name}')

winsor_df = pd.DataFrame(winsor_stats)
winsor_df.to_csv(os.path.join(output_dir, 'winsor_summary.csv'), index=False)
display(winsor_df)

## 15. 选择最终列并保存

In [ ]:
final_cols = ['code', 'year', 'company_name', 'industry_code', 'industry_cat']
final_cols += [c for c in panel.columns if c.endswith('_raw')]  # raw ratios
final_cols += [c.replace('_raw', '') for c in winsor_vars]       # winsorized ratios
final_cols += ['Top1', 'Size', 'Age']

# Keep only existing
final_cols = [c for c in final_cols if c in panel.columns]
# Deduplicate
final_cols = list(dict.fromkeys(final_cols))

panel_final = panel[final_cols].sort_values(['code', 'year']).reset_index(drop=True)

log(f'Final panel: {len(panel_final)} rows x {len(panel_final.columns)} cols')
log(f'Firms: {panel_final["code"].nunique()}, Years: {panel_final["year"].min()}-{panel_final["year"].max()}')

# Save CSV
panel_final.to_csv(os.path.join(clean_dir, 'firm_year_clean.csv'), index=False)
panel_final.to_csv(os.path.join(combined_dir, 'csmar_firm_year_panel.csv'), index=False)

# Save Parquet
panel_final.to_parquet(os.path.join(combined_dir, 'csmar_firm_year_panel.parquet'), index=False)
log('Saved CSV + Parquet')

## 16. CSV vs Parquet 对比

In [ ]:
import os as _os
import time as _time
import pyarrow.parquet as pq

csv_path = os.path.join(combined_dir, 'csmar_firm_year_panel.csv')
pq_path = os.path.join(combined_dir, 'csmar_firm_year_panel.parquet')

# 1. 只读取少数列，展示列式存储的优势
avail_cols = [c for c in ['code', 'year', 'Lev', 'ROA', 'Cash'] if c in panel_final.columns]
t0 = _time.time()
df_small = pd.read_parquet(pq_path, columns=avail_cols)
t_pq_select = _time.time() - t0
print(f'Parquet 读取 {len(avail_cols)} 列 ({len(df_small)} 行): {t_pq_select:.4f}s')

# 2. 查看 Parquet Schema
schema = pq.read_schema(pq_path)
print(f'\n=== Parquet Schema ===')
print(schema)

# 3. 比较 CSV 和 Parquet 的读取速度与文件体积
csv_size = _os.path.getsize(csv_path) / 1024
pq_size = _os.path.getsize(pq_path) / 1024

t0 = _time.time()
pd.read_csv(csv_path)
t_csv = _time.time() - t0
print(f'\nCSV 全量读取: {t_csv:.3f}s, 文件大小: {csv_size:.0f} KB')

t0 = _time.time()
pd.read_parquet(pq_path)
t_pq = _time.time() - t0
print(f'Parquet 全量读取: {t_pq:.3f}s, 文件大小: {pq_size:.0f} KB')

print(f'\n压缩比: {csv_size/pq_size:.1f}x')
print(f'列选择读取加速比: {t_csv/t_pq_select:.1f}x')
print(f'全量读取加速比: {t_csv/t_pq:.1f}x')

print(f'\n### 格式对比结论')
print(f'在本次数据规模下（约 14.6 万行 × 28 列），Parquet 文件体积为 CSV 的 1/{csv_size/pq_size:.0f}，')
print(f'列选择读取速度快约 {t_csv/t_pq_select:.0f} 倍。全量读取的速度差异不大（{t_csv/t_pq:.1f}x），')
print(f'但若扩展到全部 A 股公司季度数据（约 5000 家 × 4 季度 × 25 年 = 50 万行）')
print(f'或合并多个数据库（资产负债表 + 利润表 + 现金流量表 + 治理结构），')
print(f'Parquet 的列式存储和谓词下推优势会更加显著，速度差异可能达 100 倍以上。')

## 17. 变量字典

In [ ]:
var_defs = {
    'Lev': ('总负债率', '总负债 / 总资产', '小数'),
    'SL': ('流动负债率', '流动负债 / 总资产', '小数'),
    'LL': ('长期负债率', '非流动负债 / 总资产', '小数'),
    'SDR': ('短债比率', '流动负债 / 总负债', '小数'),
    'Cash': ('现金比率', '货币资金 / 总资产', '小数'),
    'ROA': ('总资产收益率', '净利润 / 总资产', '小数'),
    'ROE': ('净资产收益率', '净利润 / 净资产', '小数'),
    'SLoan': ('短期银行借款率', '短期借款 / 总资产', '小数'),
    'LLoan': ('长期银行借款率', '长期借款 / 总资产', '小数'),
    'Top1': ('第一大股东持股比例', '第一大股东持股比例', '小数'),
    'HHI5': ('前五大股东持股集中度', '前五大股东持股比例平方和', '小数'),
    'Size': ('公司规模', 'ln(总资产)', '对数'),
    'Age': ('上市年限', '会计年度 - 上市年份 + 1', '年'),
}

dict_rows = []
for var, (defn, formula, unit) in var_defs.items():
    dict_rows.append({
        'source_file': '资产负债表-2000-2010/2011-2024 + 利润表-2000-2010/2011-2024 + 常用变量查询 + 公司信息年度表',
        'raw_variable': f'{var}_raw' if var not in ['Top1', 'Size', 'Age'] else var,
        'clean_variable': var,
        'definition': defn,
        'unit': unit,
        'note': formula
    })

pd.DataFrame(dict_rows).to_csv(os.path.join(dict_dir, 'variable_dictionary.csv'), index=False)
log('Variable dictionary saved')

## 18. 写入处理日志

In [ ]:
log('=== 02_clean_construct_variables COMPLETE ===')
with open(os.path.join(PROJECT_ROOT, 'process_log.txt'), 'a', encoding='utf-8') as f:
    f.write('\n'.join(log_lines) + '\n')
print('Process log appended.')